# Morphological reconstruction by dilation from LoG markers

Vincent's grayscale reconstruction grows a marker image under a mask by repeated geodesic dilations until stability (Luc Vincent, *Morphological grayscale reconstruction in image analysis: applications and efficient algorithms*, IEEE TIP 2(2):176-201, 1993). Soille's *Morphological Image Analysis* (2nd ed., 2003, ch. 6) gives the standard textbook presentation.

That seeded-growth pattern is a plausible dendrite-mask candidate for these structural-channel patches. The structural channel often looks like dotted dendrites: detect reliable bright puncta with LoG, render them as marker disks, and let them expand only inside a permissive bright mask from the blurred structural image. If the mask is broken, growth stops; if faint but connected signal exists, the reconstruction bridges it. This is the same marker-under-mask logic commonly reused in seeded curvilinear segmentation in medical imaging, including sparse-marker vessel/catheter style problems.

This notebook is only a fast sanity check: eight 128x128 patches, a small parameter sweep, and visual review of one recommended working point. It does **not** generate masks for the full dataset.


## Imports + repo root


In [ ]:
import csv
import io
import itertools
import sys
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from skimage.draw import disk as draw_disk
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label as cc_label, regionprops
from skimage.morphology import dilation, disk, reconstruction, skeletonize

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / ".git").is_dir():
    REPO_ROOT = REPO_ROOT.parent
SRC_ROOT = REPO_ROOT / "src"
for p in (SRC_ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from synaptic_ssl.pseudolabels.blobs import (
    BlobPseudoCfg,
    detect_blobs_log,
    _prune_skeleton_branches,
)
from synaptic_ssl.training.sanity_batch import SANITY_TRAIN_INDICES
from synaptic_ssl.utils_data.reassemble import ImageCache, load_patch_records

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 9

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)


## Configuration

`PATCH_ROOT` is the canonical dataset location. If it is missing locally but `data/patches.tar.gz` exists, the notebook extracts only the eight demo patches into a small cache and still loads them through `load_patch_records` + `ImageCache`.


In [ ]:
PATCH_ROOT = REPO_ROOT / "data" / "patches_128"
PATCH_ARCHIVE = REPO_ROOT / "data" / "patches.tar.gz"
DEMO_PATCH_ROOT = REPO_ROOT / ".outs" / "patches_128_demo_morph_reconstruction"
EXCLUDE_PATTERNS = ["KONTROLA"]

STRUCTURAL_CHANNEL = 2
N_DEMO_PATCHES = 8
DEMO_BASE_INDICES = tuple(SANITY_TRAIN_INDICES)

LOG_BASE_CFG = dict(
    log_min_sigma=0.7,
    log_max_sigma=2.0,
    log_num_sigma=4,
    log_overlap=0.5,
    log_exclude_border=5,
)

MASK_PERCENTILES = (60, 70, 80)
LOG_THRESHOLDS = (0.003, 0.005, 0.01)
MARKER_RADIUS_SCALES = (1.0,)
PRUNE_LENGTHS = (8,)
DILATE_RADII = (2,)

PARAM_GRID = [
    dict(
        mask_percentile=mask_percentile,
        log_threshold=log_threshold,
        marker_radius_scale=marker_radius_scale,
        prune_len=prune_len,
        dilate_r=dilate_r,
    )
    for mask_percentile, log_threshold, marker_radius_scale, prune_len, dilate_r
    in itertools.product(
        MASK_PERCENTILES,
        LOG_THRESHOLDS,
        MARKER_RADIUS_SCALES,
        PRUNE_LENGTHS,
        DILATE_RADII,
    )
]

DISPLAY_PARAMS = dict(
    mask_percentile=70,
    log_threshold=0.005,
    marker_radius_scale=1.0,
    prune_len=8,
    dilate_r=2,
)

print("PATCH_ROOT      :", PATCH_ROOT, "(index:", (PATCH_ROOT / 'index.csv').exists(), ")")
print("PATCH_ARCHIVE   :", PATCH_ARCHIVE, "(exists:", PATCH_ARCHIVE.exists(), ")")
print("DEMO_PATCH_ROOT :", DEMO_PATCH_ROOT)
print("SANITY indices  :", DEMO_BASE_INDICES)
print("PARAM_GRID SIZE :", len(PARAM_GRID))
print("DISPLAY_PARAMS  :", DISPLAY_PARAMS)


## Load demo patches

The helper starts from `SANITY_TRAIN_INDICES` and pads to eight demo positions with nearby neighbours. When the extracted flat patch directory is unavailable, it writes only those eight records and `.npy` files into a local demo cache, then reuses the normal `load_patch_records` + `ImageCache` path.


In [ ]:
def filter_index_rows(rows, exclude_patterns):
    pats = [p.upper() for p in exclude_patterns]
    return [
        r
        for r in rows
        if str(r.get("damaged", "")).strip().lower() not in ("true", "1")
        and not any(p in r.get("source_image", "").upper() for p in pats)
    ]


def resolve_demo_positions(n_records, base_indices, n_target=8):
    if n_records <= 0:
        return []
    target = min(int(n_target), int(n_records))
    resolved, seen = [], set()
    max_idx = n_records - 1

    for offset in (0, 1, -1, 2, -2):
        for idx in base_indices:
            pos = min(max(int(idx) + offset, 0), max_idx)
            if pos in seen:
                continue
            resolved.append(pos)
            seen.add(pos)
            if len(resolved) == target:
                return resolved

    cursor = 0
    while len(resolved) < target and cursor <= max_idx:
        if cursor not in seen:
            resolved.append(cursor)
            seen.add(cursor)
        cursor += 1
    return resolved


def read_archive_rows(archive_path):
    with tarfile.open(archive_path, "r:gz") as tf:
        raw = tf.extractfile("data/patches_128/index.csv")
        if raw is None:
            raise FileNotFoundError("data/patches_128/index.csv not found inside archive.")
        text = io.TextIOWrapper(raw, encoding="utf-8")
        return list(csv.DictReader(text))


def extract_demo_subset(archive_path, out_root, exclude_patterns, base_indices, n_target):
    if not archive_path.exists():
        raise FileNotFoundError(f"Missing archive: {archive_path}")

    rows = read_archive_rows(archive_path)
    filtered = filter_index_rows(rows, exclude_patterns)
    demo_positions = resolve_demo_positions(len(filtered), base_indices, n_target=n_target)
    subset = []
    for pos in demo_positions:
        row = filtered[pos].copy()
        row["original_pos"] = int(pos)
        subset.append(row)

    out_root.mkdir(parents=True, exist_ok=True)
    wanted = {row["filename"] for row in subset}
    for stale in out_root.glob("*.npy"):
        if stale.name not in wanted:
            stale.unlink()

    with tarfile.open(archive_path, "r:gz") as tf:
        for row in subset:
            dst = out_root / row["filename"]
            if dst.exists():
                continue
            member = f"data/patches_128/{row['filename']}"
            src = tf.extractfile(member)
            if src is None:
                raise FileNotFoundError(f"Missing {member} inside archive.")
            with dst.open("wb") as fh:
                fh.write(src.read())

    fieldnames = list(subset[0].keys())
    with (out_root / "index.csv").open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(subset)
    return out_root


use_full_patch_root = False
if (PATCH_ROOT / "index.csv").exists():
    full_patch_records = load_patch_records(PATCH_ROOT, EXCLUDE_PATTERNS)
    full_demo_positions = resolve_demo_positions(
        len(full_patch_records),
        DEMO_BASE_INDICES,
        n_target=N_DEMO_PATCHES,
    )
    needed_image_ids = {
        int(full_patch_records[pos]["image_index"])
        for pos in full_demo_positions
    }
    needed_files = [
        PATCH_ROOT / rec["filename"]
        for rec in full_patch_records
        if int(rec["image_index"]) in needed_image_ids
    ]
    use_full_patch_root = bool(needed_files) and all(path.exists() for path in needed_files)
    print(
        "PATCH_ROOT complete for demo images:",
        use_full_patch_root,
        f"({sum(path.exists() for path in needed_files)}/{len(needed_files)} files present)",
    )

if use_full_patch_root:
    ACTIVE_PATCH_ROOT = PATCH_ROOT
    patch_records = full_patch_records
    demo_positions = full_demo_positions
else:
    ACTIVE_PATCH_ROOT = extract_demo_subset(
        PATCH_ARCHIVE,
        DEMO_PATCH_ROOT,
        EXCLUDE_PATTERNS,
        DEMO_BASE_INDICES,
        N_DEMO_PATCHES,
    )
    patch_records = load_patch_records(ACTIVE_PATCH_ROOT, EXCLUDE_PATTERNS)
    demo_positions = list(range(len(patch_records)))

cache = ImageCache(ACTIVE_PATCH_ROOT, patch_records)
demo_items = []
for pos in demo_positions:
    patch = cache.get_patch(pos).astype(np.float32)
    rec = patch_records[pos]
    demo_items.append(
        dict(
            pos=pos,
            original_pos=int(rec.get("original_pos", pos)),
            filename=rec["filename"],
            source_image=rec.get("source_image", ""),
            image_index=int(rec["image_index"]),
            grid_row=int(rec["grid_row"]),
            grid_col=int(rec["grid_col"]),
            patch=patch,
            structural=patch[STRUCTURAL_CHANNEL].astype(np.float32, copy=False),
        )
    )

print("ACTIVE_PATCH_ROOT:", ACTIVE_PATCH_ROOT)
print("Loaded demo patches:", len(demo_items))
for item in demo_items:
    print(
        f"orig={item['original_pos']:>4d}  subset={item['pos']:>2d}  "
        f"img={item['image_index']}  rc=({item['grid_row']},{item['grid_col']})  {item['filename']}"
    )

assert all(item["patch"].shape == (3, 128, 128) for item in demo_items)


## Reconstruction helpers

`detect_markers_log` reuses `detect_blobs_log` on the structural channel and renders each detection as a disk of radius `round(marker_radius_scale * sqrt(2) * sigma)`. `build_mask_image` applies the permissive low-percentile threshold after a small Gaussian blur. `reconstruct_dendrite` runs Vincent's reconstruction-by-dilation, and `postprocess` follows the requested skeletonise → prune short branches → re-dilate recipe.


In [ ]:
def make_log_cfg(log_threshold):
    return BlobPseudoCfg(
        structural_channel=STRUCTURAL_CHANNEL,
        log_threshold=float(log_threshold),
        **LOG_BASE_CFG,
    )


def detect_markers_log(structural, cfg, marker_radius_scale=1.0, return_blobs=False):
    structural = np.asarray(structural, dtype=np.float32)
    blobs = detect_blobs_log(structural, cfg)
    markers = np.zeros_like(structural, dtype=np.float32)
    for row, col, sigma in blobs:
        radius = max(
            1,
            int(round(float(marker_radius_scale) * np.sqrt(2.0) * float(sigma))),
        )
        rr, cc = draw_disk(
            (int(round(row)), int(round(col))),
            radius,
            shape=structural.shape,
        )
        markers[rr, cc] = 1.0
    if return_blobs:
        return markers, blobs
    return markers


def build_mask_image(structural, percentile, return_meta=False):
    structural = np.asarray(structural, dtype=np.float32)
    blurred = gaussian(structural, sigma=1.0, preserve_range=True).astype(np.float32)
    threshold = float(np.percentile(blurred, float(percentile)))
    mask = np.where(blurred >= threshold, blurred, 0.0).astype(np.float32)
    if return_meta:
        return mask, dict(blurred=blurred, threshold=threshold)
    return mask


def reconstruct_dendrite(markers, mask):
    markers = np.asarray(markers, dtype=np.float32)
    mask = np.asarray(mask, dtype=np.float32)
    seed = np.where(markers > 0, mask, 0.0).astype(np.float32)
    seed = np.minimum(seed, mask)
    if not seed.any() or not mask.any():
        return np.zeros_like(mask, dtype=np.float32)
    return reconstruction(seed, mask, method="dilation").astype(np.float32)


def postprocess(reconstructed, prune_len, dilate_r, return_meta=False):
    reconstructed = np.asarray(reconstructed, dtype=np.float32)
    positive = reconstructed[reconstructed > 0]
    if positive.size == 0:
        zero = np.zeros_like(reconstructed, dtype=bool)
        if return_meta:
            return zero, dict(
                threshold=np.nan,
                binary=zero,
                skeleton=zero,
                pruned=zero,
            )
        return zero

    if positive.size > 1 and float(positive.max() - positive.min()) > 0:
        threshold = float(threshold_otsu(positive))
    else:
        threshold = float(positive.min())

    binary = reconstructed >= threshold
    skeleton = skeletonize(binary)
    pruned = _prune_skeleton_branches(skeleton, max_len=int(prune_len))
    dilate_r = max(0, int(dilate_r))
    final_mask = dilation(pruned, disk(dilate_r)) if dilate_r > 0 and pruned.any() else pruned
    final_mask = np.asarray(final_mask, dtype=bool)

    if return_meta:
        return final_mask, dict(
            threshold=threshold,
            binary=np.asarray(binary, dtype=bool),
            skeleton=np.asarray(skeleton, dtype=bool),
            pruned=np.asarray(pruned, dtype=bool),
        )
    return final_mask


def run_configuration(structural, params):
    log_cfg = make_log_cfg(params["log_threshold"])
    markers, blobs = detect_markers_log(
        structural,
        log_cfg,
        marker_radius_scale=params["marker_radius_scale"],
        return_blobs=True,
    )
    mask, mask_meta = build_mask_image(
        structural,
        params["mask_percentile"],
        return_meta=True,
    )
    reconstructed = reconstruct_dendrite(markers, mask)
    final_mask, post_meta = postprocess(
        reconstructed,
        prune_len=params["prune_len"],
        dilate_r=params["dilate_r"],
        return_meta=True,
    )
    return dict(
        params=dict(params),
        blobs=blobs,
        markers=markers,
        mask=mask,
        blurred=mask_meta["blurred"],
        mask_threshold=mask_meta["threshold"],
        reconstructed=reconstructed,
        reconstructed_threshold=post_meta["threshold"],
        binary=post_meta["binary"],
        skeleton=post_meta["skeleton"],
        pruned=post_meta["pruned"],
        final_mask=final_mask,
    )


def summarize_result(result):
    lbl = cc_label(result["final_mask"])
    props = regionprops(lbl)
    return dict(
        n_blobs=int(len(result["blobs"])),
        mask_fraction=float((result["mask"] > 0).mean()),
        reconstructed_fraction=float((result["reconstructed"] > 0).mean()),
        final_fraction=float(result["final_mask"].mean()),
        n_cc=int(lbl.max()),
        largest_cc=int(max((prop.area for prop in props), default=0)),
    )


def normalize_for_display(image):
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(image, (2, 99.5))
    if hi <= lo:
        base = np.zeros_like(image, dtype=np.float32)
    else:
        base = np.clip((image - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    return np.dstack([base, base, base])


def overlay_mask_on_raw(structural, mask, color=(1.0, 0.2, 0.2), alpha=0.55):
    rgb = normalize_for_display(structural)
    mask = np.asarray(mask, dtype=bool)
    rgb[mask] = (1.0 - alpha) * rgb[mask] + alpha * np.asarray(color, dtype=np.float32)
    return rgb


## Parameter sweep

The sweep keeps the marker radius scale, prune length, and final dilation fixed at a conservative setting and varies the two main knobs requested here: the permissive mask percentile and the LoG threshold on the structural channel. Without labels there is no exact objective, so the notebook reports simple patch-level diagnostics averaged over the eight demo patches.


In [ ]:
sweep_rows = []
per_config_results = {}

for params in PARAM_GRID:
    outputs = [run_configuration(item["structural"], params) for item in demo_items]
    summaries = [summarize_result(output) for output in outputs]
    key = (
        float(params["mask_percentile"]),
        float(params["log_threshold"]),
        float(params["marker_radius_scale"]),
        int(params["prune_len"]),
        int(params["dilate_r"]),
    )
    per_config_results[key] = outputs
    sweep_rows.append(
        dict(
            mask_percentile=float(params["mask_percentile"]),
            log_threshold=float(params["log_threshold"]),
            marker_radius_scale=float(params["marker_radius_scale"]),
            prune_len=int(params["prune_len"]),
            dilate_r=int(params["dilate_r"]),
            mean_blobs=float(np.mean([s["n_blobs"] for s in summaries])),
            mean_mask_fraction=float(np.mean([s["mask_fraction"] for s in summaries])),
            mean_reconstructed_fraction=float(np.mean([s["reconstructed_fraction"] for s in summaries])),
            mean_final_fraction=float(np.mean([s["final_fraction"] for s in summaries])),
            mean_cc=float(np.mean([s["n_cc"] for s in summaries])),
            mean_largest_cc=float(np.mean([s["largest_cc"] for s in summaries])),
        )
    )

sweep_rows = sorted(sweep_rows, key=lambda row: (row["mask_percentile"], row["log_threshold"]))
header = (
    f"{'mask_p':>6}  {'log_thr':>7}  {'mean_blobs':>10}  "
    f"{'mask%':>8}  {'recon%':>8}  {'final%':>8}  {'mean_cc':>8}  {'largest_cc':>11}"
)
print(header)
print("-" * len(header))
for row in sweep_rows:
    print(
        f"{row['mask_percentile']:6.0f}  "
        f"{row['log_threshold']:7.3f}  "
        f"{row['mean_blobs']:10.1f}  "
        f"{100 * row['mean_mask_fraction']:7.2f}%  "
        f"{100 * row['mean_reconstructed_fraction']:7.2f}%  "
        f"{100 * row['mean_final_fraction']:7.2f}%  "
        f"{row['mean_cc']:8.2f}  "
        f"{row['mean_largest_cc']:11.2f}"
    )

recommended_row = next(
    row
    for row in sweep_rows
    if row["mask_percentile"] == float(DISPLAY_PARAMS["mask_percentile"])
    and abs(row["log_threshold"] - float(DISPLAY_PARAMS["log_threshold"])) < 1e-12
)
print()
print("Working recommendation:", DISPLAY_PARAMS)
print(
    "Observation:",
    "mask_percentile shifts support strongly, while log_threshold=0.003..0.01 barely changes the result on these eight patches.",
)
recommended_row


In [ ]:
final_grid = np.full((len(MASK_PERCENTILES), len(LOG_THRESHOLDS)), np.nan, dtype=float)
cc_grid = np.full((len(MASK_PERCENTILES), len(LOG_THRESHOLDS)), np.nan, dtype=float)

for row in sweep_rows:
    i = list(MASK_PERCENTILES).index(int(row["mask_percentile"]))
    j = list(LOG_THRESHOLDS).index(float(row["log_threshold"]))
    final_grid[i, j] = 100.0 * row["mean_final_fraction"]
    cc_grid[i, j] = row["mean_cc"]


def annotate_heatmap(ax, data, fmt):
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(
                j,
                i,
                format(data[i, j], fmt),
                ha="center",
                va="center",
                color="white" if data[i, j] >= np.nanmean(data) else "black",
                fontsize=9,
            )


fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

im0 = axes[0].imshow(final_grid, cmap="magma", aspect="auto")
axes[0].set_title("mean final foreground (%)")
axes[0].set_xticks(range(len(LOG_THRESHOLDS)), [f"{v:.3f}" for v in LOG_THRESHOLDS])
axes[0].set_yticks(range(len(MASK_PERCENTILES)), [f"P{v}" for v in MASK_PERCENTILES])
axes[0].set_xlabel("LoG threshold")
axes[0].set_ylabel("mask percentile")
annotate_heatmap(axes[0], final_grid, ".1f")
fig.colorbar(im0, ax=axes[0], shrink=0.82)

im1 = axes[1].imshow(cc_grid, cmap="viridis", aspect="auto")
axes[1].set_title("mean connected components")
axes[1].set_xticks(range(len(LOG_THRESHOLDS)), [f"{v:.3f}" for v in LOG_THRESHOLDS])
axes[1].set_yticks(range(len(MASK_PERCENTILES)), [f"P{v}" for v in MASK_PERCENTILES])
axes[1].set_xlabel("LoG threshold")
axes[1].set_ylabel("mask percentile")
annotate_heatmap(axes[1], cc_grid, ".1f")
fig.colorbar(im1, ax=axes[1], shrink=0.82)

fig.suptitle("Sweep summary over 8 demo patches", y=1.03)
plt.show()


## Eight-patch visual sanity check

Rows are demo patches. Columns show the raw structural channel, LoG marker disks, the permissive mask image, the grayscale reconstruction, the final binary dendrite mask, and the final mask overlaid on the structural channel.


In [ ]:
display_key = (
    float(DISPLAY_PARAMS["mask_percentile"]),
    float(DISPLAY_PARAMS["log_threshold"]),
    float(DISPLAY_PARAMS["marker_radius_scale"]),
    int(DISPLAY_PARAMS["prune_len"]),
    int(DISPLAY_PARAMS["dilate_r"]),
)
display_results = per_config_results[display_key]

fig, axes = plt.subplots(
    len(demo_items),
    6,
    figsize=(18, 2.7 * len(demo_items)),
    constrained_layout=True,
)
if len(demo_items) == 1:
    axes = np.asarray([axes])

col_titles = [
    "raw structural",
    "LoG markers overlay",
    "mask image",
    "reconstructed",
    "final dendrite mask",
    "overlay",
]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title)

for row, (item, result) in enumerate(zip(demo_items, display_results)):
    structural = item["structural"]
    metrics = summarize_result(result)
    vmax = float(np.quantile(structural, 0.995)) if structural.size else 1.0
    vmax = max(vmax, 1e-6)
    rec_thr = result["reconstructed_threshold"]
    rec_thr_text = "nan" if np.isnan(rec_thr) else f"{rec_thr:.3g}"

    axes[row, 0].imshow(structural, cmap="gray", vmin=0.0, vmax=vmax)
    axes[row, 1].imshow(
        overlay_mask_on_raw(
            structural,
            result["markers"] > 0,
            color=(1.0, 0.85, 0.1),
            alpha=0.65,
        )
    )
    axes[row, 2].imshow(result["mask"], cmap="gray")
    axes[row, 3].imshow(result["reconstructed"], cmap="magma")
    axes[row, 4].imshow(result["final_mask"], cmap="gray", interpolation="nearest")
    axes[row, 5].imshow(overlay_mask_on_raw(structural, result["final_mask"]))

    axes[row, 0].set_ylabel(
        f"idx {item['original_pos']}\nimg {item['image_index']}\n({item['grid_row']},{item['grid_col']})",
        rotation=0,
        ha="right",
        va="center",
        labelpad=30,
    )
    axes[row, 1].text(
        0.02,
        0.98,
        f"blobs={metrics['n_blobs']}",
        transform=axes[row, 1].transAxes,
        va="top",
        ha="left",
        fontsize=7,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.45, "pad": 2, "edgecolor": "none"},
    )
    axes[row, 3].text(
        0.02,
        0.98,
        f"mask>{result['mask_threshold']:.3g}\nrec_otsu={rec_thr_text}",
        transform=axes[row, 3].transAxes,
        va="top",
        ha="left",
        fontsize=7,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.45, "pad": 2, "edgecolor": "none"},
    )
    axes[row, 5].text(
        0.02,
        0.98,
        f"final={100 * metrics['final_fraction']:.1f}%\ncc={metrics['n_cc']}\nmaxCC={metrics['largest_cc']}",
        transform=axes[row, 5].transAxes,
        va="top",
        ha="left",
        fontsize=7,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.45, "pad": 2, "edgecolor": "none"},
    )

    for col in range(6):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

fig.suptitle(
    "Morphological reconstruction from structural-channel LoG markers\n"
    f"mask_percentile={DISPLAY_PARAMS['mask_percentile']}, "
    f"log_threshold={DISPLAY_PARAMS['log_threshold']:.3f}, "
    f"marker_radius_scale={DISPLAY_PARAMS['marker_radius_scale']}, "
    f"prune_len={DISPLAY_PARAMS['prune_len']}, dilate_r={DISPLAY_PARAMS['dilate_r']}",
    y=1.02,
    fontsize=12,
)
plt.show()


## Takeaways

- **Recommended starting point:** `mask_percentile=70`, `log_threshold=0.005`, `marker_radius_scale=1.0`, `prune_len=8`, `dilate_r=2`.
- **Why this setting:** on this eight-patch sweep, `P60` grows more aggressively through haze, `P80` breaks many faint bridges, and `P70` keeps the most plausible connections without flooding the patch. Within `log_threshold=0.003..0.01`, the marker count changes only slightly here, so the mask percentile is the practical aggressiveness knob.
- **Failure modes:** (i) low percentile merges background haze into false bridges; (ii) high percentile snaps faint dendrites into short fragments; (iii) textured structural background can seed short false branches if LoG is too permissive; (iv) if the ceiling mask is genuinely broken, reconstruction will not invent a connection.
- **Compared with the existing density pipeline:** the density route smooths a puncta/intensity field and thresholds that response. Reconstruction is more explicit marker-under-mask growth: disconnected puncta clusters should stay disconnected, while weak but connected structural signal can still bridge puncta without tuning a large KDE bandwidth or density sigma.

As a next comparison point, it would be worth placing this notebook's `P70 / 0.005` working point next to the current density-based dendrite masks on the same eight demo patches and checking whether reconstruction suppresses isolated puncta clouds more reliably.
